# 04 - Data Cleaning

**Input:** `data/interim/train.csv` | `val.csv` | `test.csv` (DVC-tracked)

**Output:** `data/processed/train_clean.csv` | `val_clean.csv` | `test_clean.csv` (DVC-tracked)

**Config:** `configs/data_config.yaml` -> `eda_derived` section

---

### What this notebook does

| Step | Action | Source |
|---|---|---|
| 1 | Global median imputation on `total_bedrooms` | MCAR confirmed (chi2 p=0.3095) |
| 2 | Add `is_capped` flag | target >= $500,001 |
| 3 | log1p on count columns | EDA skewness analysis |
| 4 | LOF outlier flag | contamination=0.02 from histogram elbow |
| 5 | Save to `data/processed/` | DVC-tracked |

### FIT/TRANSFORM rule

All statistics (median, LOF model) are **fit on train only** and applied to val/test.
This prevents data leakage from test information into training decisions.

---
## 0 - Setup & Load

In [1]:
import os
import sys
from pathlib import Path

repo_path = Path("/content/california_housing_full_project")
os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

(repo_path / "src" / "__init__.py").touch(exist_ok=True)
(repo_path / "src" / "data" / "__init__.py").touch(exist_ok=True)

import importlib
importlib.invalidate_caches()

print(f"✅ Working dir : {os.getcwd()}")
print(f"✅ sys.path[0] : {sys.path[0]}")

✅ Working dir : /content/california_housing_full_project
✅ sys.path[0] : /content/california_housing_full_project


---
## 1 - Imports

In [ ]:
import logging
import pandas as pd
import numpy as np

from src.utils.logger import setup_logging, get_logger
from src.data.data_loader import DataLoader
from src.data.cleaning import (
    run_cleaning,
    load_eda_config,
    CleaningResult,
    CleaningError,
)

setup_logging(level=logging.INFO)
logger = get_logger("notebook.04_cleaning")

CONFIG_PATH = "configs/data_config.yaml"

print("Imports ready")

Imports ready


---
## 2 - Pull Interim Splits from DVC

The interim splits (`train.csv`, `val.csv`, `test.csv`) were created by `02_splitting.ipynb`
and tracked with DVC. We pull them here before loading.

In [17]:
import os
import subprocess
from pathlib import Path
from src.utils.paths import IN_COLAB

# Try to get the configured remote name, fallback to 'mylocal' if not defined
try:
    from src.utils.paths import DVC_REMOTE_NAME
except ImportError:
    DVC_REMOTE_NAME = "mylocal"
    print(f"⚠️ DVC_REMOTE_NAME not found, falling back to '{DVC_REMOTE_NAME}'")

# Define the paths to check
interim_files = ["data/interim/train.csv", "data/interim/val.csv", "data/interim/test.csv"]
all_exist = all(Path(f).exists() for f in interim_files)

if all_exist:
    print("✅ Interim files already present locally. Skipping DVC pull.")
else:
    if IN_COLAB:
        print(f"🔄 Running DVC pull from remote: {DVC_REMOTE_NAME}...")
        result = subprocess.run(
            ["dvc", "pull", f"--remote={DVC_REMOTE_NAME}", "data/interim"],
            capture_output=True, text=True, cwd=os.getcwd()
        )
        if result.returncode == 0:
            print("✅ DVC pull successful")
        else:
            print(f"⚠️ DVC pull warning: {result.stderr.strip()}")
            print("If this is the first run, data may already be present locally.")
    else:
        print("Local environment - skipping DVC pull (data should already be present)")

# Verify files exist
print("\n-- Verification --")
for f in interim_files:
    status = "✅ OK" if Path(f).exists() else "❌ MISSING"
    size_mb = Path(f).stat().st_size / 1024**2 if Path(f).exists() else 0
    print(f"  {status}  {f}  ({size_mb:.2f} MB)")

✅ Interim files already present locally. Skipping DVC pull.

-- Verification --
  ✅ OK  data/interim/train.csv  (0.95 MB)
  ✅ OK  data/interim/val.csv  (0.20 MB)
  ✅ OK  data/interim/test.csv  (0.20 MB)


---
## 3 - Load Interim Splits

In [6]:
loader = DataLoader()

train = loader.load_interim("train.csv")
val   = loader.load_interim("val.csv")
test  = loader.load_interim("test.csv")

print(f"train : {train.shape[0]:,} rows x {train.shape[1]} cols")
print(f"val   : {val.shape[0]:,} rows x {val.shape[1]} cols")
print(f"test  : {test.shape[0]:,} rows x {test.shape[1]} cols")
print()
print("Null counts (train):")
nulls = train.isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.sum() > 0 else "  No nulls")

2026-06-18 21:15:23 | INFO     | src.data.data_loader | DataLoader initialized | Drive mode: True
2026-06-18 21:15:23 | INFO     | src.data.data_loader | Loading: /content/california_housing_full_project/data/interim/train.csv
2026-06-18 21:15:24 | INFO     | src.data.data_loader | Loaded 'train.csv' | shape=(14448, 10) | stage=interim
2026-06-18 21:15:24 | INFO     | src.data.data_loader | Loading: /content/california_housing_full_project/data/interim/val.csv
2026-06-18 21:15:24 | INFO     | src.data.data_loader | Loaded 'val.csv' | shape=(3096, 10) | stage=interim
2026-06-18 21:15:24 | INFO     | src.data.data_loader | Loading: /content/california_housing_full_project/data/interim/test.csv
2026-06-18 21:15:24 | INFO     | src.data.data_loader | Loaded 'test.csv' | shape=(3096, 10) | stage=interim
train : 14,448 rows x 10 cols
val   : 3,096 rows x 10 cols
test  : 3,096 rows x 10 cols

Null counts (train):
total_bedrooms    140


---
## 4 - Verify EDA Config

Before cleaning, confirm that `data_config.yaml` has the `eda_derived` section
with real values from `notebooks/03_eda.ipynb`. If `source = fallback`, stop and
update the config first.

In [7]:
eda_cfg = load_eda_config(CONFIG_PATH)

print(f"EDA config source    : {eda_cfg.source}")
print(f"log1p columns        : {eda_cfg.log1p_columns}")
print(f"cap threshold        : ${eda_cfg.cap_threshold:,.0f}")
print(f"LOF contamination    : {eda_cfg.lof_contamination}")
print(f"LOF n_neighbors      : {eda_cfg.lof_n_neighbors}")
print(f"LOF features         : {eda_cfg.lof_features}")
print(f"Imputation strategy  : {eda_cfg.imputation_strategy}")
print()

if eda_cfg.source == "fallback":
    print("WARNING: eda_derived section missing from data_config.yaml!")
    print("Run notebooks/03_eda.ipynb extraction cell and update configs/data_config.yaml")
    print("before proceeding.")
else:
    print("Config loaded from data_config.yaml - ready to clean")

2026-06-18 21:15:43 | INFO     | src.data.cleaning | EDA config loaded: log1p_cols=['total_rooms', 'total_bedrooms', 'population', 'households'], cap_threshold=500001.0, lof_contamination=0.02
EDA config source    : config
log1p columns        : ['total_rooms', 'total_bedrooms', 'population', 'households']
cap threshold        : $500,001
LOF contamination    : 0.02
LOF n_neighbors      : 20
LOF features         : ['median_income', 'total_rooms', 'population', 'households', 'longitude', 'latitude']
Imputation strategy  : global_median

Config loaded from data_config.yaml - ready to clean


---
## 5 - Run Cleaning Pipeline

`run_cleaning()` applies all steps in order:
1. Fit imputer on train -> apply to all three splits
2. Add `is_capped` flag
3. Apply log1p to count columns (train-fit is not needed - pure math transform)
4. Fit LOF on train -> apply outlier flag to all three splits
5. Save to `data/processed/`
6. Track with DVC

In [18]:
try:
    result = run_cleaning(
        train=train,
        val=val,
        test=test,
        config_path=CONFIG_PATH,
        auto_track_dvc=True,
        save_artifacts_flag=True,
    )
    print(result.summary())
except CleaningError as e:
    logger.error(f"❌ Cleaning pipeline failed: {e}")
    print(f"ERROR: Cleaning failed - {e}")
    raise 
except Exception as e:
    logger.error(f"❌ Unexpected error during cleaning: {e}")
    print(f"UNEXPECTED ERROR: {e}")
    raise

2026-06-18 21:37:11 | INFO     | src.data.cleaning | ============================================================
2026-06-18 21:37:11 | INFO     | src.data.cleaning |   Data cleaning started
2026-06-18 21:37:11 | INFO     | src.data.cleaning | ============================================================
2026-06-18 21:37:11 | INFO     | src.data.cleaning | EDA config loaded: log1p_cols=['total_rooms', 'total_bedrooms', 'population', 'households'], cap_threshold=500001.0, lof_contamination=0.02
2026-06-18 21:37:11 | INFO     | src.data.cleaning | Step 1/4 - Imputation
2026-06-18 21:37:11 | INFO     | src.data.cleaning | Imputer fit: 'total_bedrooms' -> median = 432.0000


2026-06-18 21:37:11 | INFO     | src.data.cleaning | Imputed 'total_bedrooms': filled 140 nulls with 432.0
2026-06-18 21:37:11 | INFO     | src.data.cleaning | Imputed 'total_bedrooms': filled 33 nulls with 432.0
2026-06-18 21:37:11 | INFO     | src.data.cleaning | Imputed 'total_bedrooms': filled 34 nulls with 432.0
2026-06-18 21:37:11 | INFO     | src.data.cleaning | Step 2/4 - is_capped flag
2026-06-18 21:37:11 | INFO     | src.data.cleaning | is_capped flag added (threshold=500001.0): 683 rows (4.73%)
2026-06-18 21:37:11 | INFO     | src.data.cleaning | is_capped flag added (threshold=500001.0): 151 rows (4.88%)
2026-06-18 21:37:11 | INFO     | src.data.cleaning | is_capped flag added (threshold=500001.0): 131 rows (4.23%)
2026-06-18 21:37:11 | INFO     | src.data.cleaning | Step 3/4 - log1p transform (count columns only)
2026-06-18 21:37:11 | INFO     | src.data.cleaning | log1p 'total_rooms': skew 4.264 -> -1.134
2026-06-18 21:37:11 | INFO     | src.data.cleaning | log1p 'total_b

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


2026-06-18 21:37:12 | INFO     | src.data.cleaning | LOF flag applied: 237 outliers, 0 unknown (nulls in features)
2026-06-18 21:37:12 | INFO     | src.data.cleaning | LOF flag applied: 59 outliers, 0 unknown (nulls in features)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


2026-06-18 21:37:12 | INFO     | src.data.cleaning | LOF flag applied: 49 outliers, 0 unknown (nulls in features)
2026-06-18 21:37:12 | INFO     | src.data.cleaning | Artifacts saved -> /content/california_housing_full_project/artifacts
2026-06-18 21:37:13 | INFO     | src.data.cleaning | Saved train_clean.csv -> /content/california_housing_full_project/data/processed/train_clean.csv (1.64 MB)
2026-06-18 21:37:13 | INFO     | src.data.cleaning | Saved val_clean.csv -> /content/california_housing_full_project/data/processed/val_clean.csv (0.35 MB)
2026-06-18 21:37:13 | INFO     | src.data.cleaning | Saved test_clean.csv -> /content/california_housing_full_project/data/processed/test_clean.csv (0.35 MB)
2026-06-18 21:37:13 | INFO     | src.data.cleaning | DVC add: data/processed
2026-06-18 21:37:14 | INFO     | src.data.cleaning | DVC push -> mylocal
2026-06-18 21:37:16 | WARNING  | src.data.cleaning | Command failed: git commit -m data: track processed cleaned splits with DVC
  stderr: 

---
## 6 - Verify Cleaning Results

Three checks:
- **Nulls removed** - `total_bedrooms` should have zero nulls
- **New columns added** - `is_capped` and `lof_outlier` must exist
- **log1p applied** - count columns should be in log scale (values < original)

In [9]:
print("-- Null check after cleaning --")
for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
    nulls = df.isnull().sum().sum()
    status = "OK" if nulls == 0 else f"FAIL ({nulls} nulls remaining)"
    print(f"  {name:<6} : {status}")

-- Null check after cleaning --
  train  : OK
  val    : OK
  test   : OK


In [10]:
print("-- New columns check --")
for col in ["is_capped", "lof_outlier"]:
    for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
        present = col in df.columns
        print(f"  {col:<15} in {name:<6} : {'OK' if present else 'MISSING'}")

print()
print("-- is_capped distribution --")
for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
    n_capped = df["is_capped"].sum()
    print(f"  {name:<6} : {n_capped:,} capped rows ({n_capped/len(df):.2%})")

print()
print("-- lof_outlier distribution --")
for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
    counts = df["lof_outlier"].value_counts().sort_index()
    print(f"  {name:<6} : {dict(counts)} (-99=unknown, 0=inlier, 1=outlier)")

-- New columns check --
  is_capped       in train  : OK
  is_capped       in val    : OK
  is_capped       in test   : OK
  lof_outlier     in train  : OK
  lof_outlier     in val    : OK
  lof_outlier     in test   : OK

-- is_capped distribution --
  train  : 683 capped rows (4.73%)
  val    : 151 capped rows (4.88%)
  test   : 131 capped rows (4.23%)

-- lof_outlier distribution --
  train  : {0: np.int64(14211), 1: np.int64(237)} (-99=unknown, 0=inlier, 1=outlier)
  val    : {0: np.int64(3037), 1: np.int64(59)} (-99=unknown, 0=inlier, 1=outlier)
  test   : {0: np.int64(3047), 1: np.int64(49)} (-99=unknown, 0=inlier, 1=outlier)


In [11]:
print("-- log1p sanity check (values should be in log scale) --")
log1p_cols = eda_cfg.log1p_columns
original_ranges = {
    "total_rooms": (500, 8000),
    "total_bedrooms": (100, 1500),
    "population": (200, 3000),
    "households": (80, 1200),
}

for col in log1p_cols:
    if col in result.train.columns:
        col_max = result.train[col].max()
        col_mean = result.train[col].mean()
        # log1p(8000) ~ 9.0 -- if values are in log scale, max should be < 15
        scale = "log scale" if col_max < 15 else "RAW scale (log1p may not have applied!)"
        print(f"  {col:<20} max={col_max:.2f}  mean={col_mean:.2f}  -> {scale}")

-- log1p sanity check (values should be in log scale) --
  total_rooms          max=10.58  mean=7.62  -> log scale
  total_bedrooms       max=8.77  mean=6.05  -> log scale
  population           max=10.48  mean=7.02  -> log scale
  households           max=8.71  mean=5.98  -> log scale


In [12]:
print("-- median_income: should be unchanged (NOT log1p transformed) --")
for name, df in [("train", result.train), ("val", result.val)]:
    income_max = df["median_income"].max()
    income_min = df["median_income"].min()
    # Original data range is 0.5-15.0; if unchanged, max should be > 5
    status = "OK (not transformed)" if income_max > 5 else "WARN (may have been transformed)"
    print(f"  {name:<6} : min={income_min:.2f}  max={income_max:.2f}  -> {status}")

-- median_income: should be unchanged (NOT log1p transformed) --
  train  : min=0.50  max=15.00  -> OK (not transformed)
  val    : min=0.50  max=15.00  -> OK (not transformed)


---
## 7 - Shape & Schema Consistency

In [13]:
print("-- Shape check --")
for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
    print(f"  {name:<6} : {df.shape[0]:,} rows x {df.shape[1]} cols")

print()
print("-- Column consistency --")
train_cols = list(result.train.columns)
for name, df in [("val", result.val), ("test", result.test)]:
    if list(df.columns) == train_cols:
        print(f"  {name} columns match train")
    else:
        diff = set(train_cols) ^ set(df.columns)
        print(f"  {name} column mismatch: {diff}")

print()
print("-- Final column list --")
print(result.train.columns.tolist())

-- Shape check --
  train  : 14,448 rows x 12 cols
  val    : 3,096 rows x 12 cols
  test   : 3,096 rows x 12 cols

-- Column consistency --
  val columns match train
  test columns match train

-- Final column list --
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity', 'is_capped', 'lof_outlier']


---
## 8 - DVC Tracking Confirmation

In [14]:
from pathlib import Path

print("-- DVC pointer files --")
for f in ["data/processed.dvc", "data/.gitignore"]:
    exists = Path(f).exists()
    print(f"  {'OK' if exists else 'MISSING'}  {f}")

print()
print("-- Saved files --")
for f in sorted(Path("data/processed").glob("*.csv")):
    size_mb = f.stat().st_size / 1024**2
    print(f"  {f.name:<22}  {size_mb:.2f} MB")

print()
print(f"DVC tracked: {'yes' if result.dvc_tracked else 'no (run dvc push manually)'}")

if result.warnings:
    print()
    print("Warnings:")
    for w in result.warnings:
        print(f"  ! {w}")

-- DVC pointer files --
  OK  data/processed.dvc
  OK  data/.gitignore

-- Saved files --
  test_clean.csv          0.35 MB
  train_clean.csv         1.64 MB
  val_clean.csv           0.35 MB

DVC tracked: yes


---
## 9 - Artifacts Check

`artifacts/cleaning_artifacts.json` stores the train-fit statistics needed
to apply the exact same transformations to new production data.

In [15]:
import json
from pathlib import Path

artifacts_path = Path("artifacts/cleaning_artifacts.json")
lof_pkl_path   = Path("artifacts/lof_model.pkl")

print("-- Artifact files --")
for f in [artifacts_path, lof_pkl_path]:
    status = "OK" if f.exists() else "MISSING"
    print(f"  {status}  {f}")

if artifacts_path.exists():
    meta = json.loads(artifacts_path.read_text())
    print()
    print("-- Artifact contents --")
    print(f"  imputer_stats    : {meta['imputer_stats']}")
    print(f"  log1p_cols       : {meta['log1p_cols']}")
    print(f"  cap_threshold    : {meta['cap_threshold']}")
    print(f"  lof_contamination: {meta['lof_contamination']}")
    print(f"  lof_n_neighbors  : {meta['lof_n_neighbors']}")
    print(f"  eda_config_source: {meta['eda_config_source']}")
    print(f"  timestamp        : {meta['timestamp']}")

-- Artifact files --
  OK  artifacts/cleaning_artifacts.json
  OK  artifacts/lof_model.pkl

-- Artifact contents --
  imputer_stats    : {'total_bedrooms': 432.0}
  log1p_cols       : ['total_rooms', 'total_bedrooms', 'population', 'households']
  cap_threshold    : 500001.0
  lof_contamination: 0.02
  lof_n_neighbors  : 20
  eda_config_source: config
  timestamp        : 2026-06-18T21:16:45.939467


---
## 10 - Git Commit

Only the `.dvc` pointer and `.gitignore` go to GitHub - never the CSV files.

In [16]:
print("Run in terminal:")
print()
print("  git add data/processed.dvc data/.gitignore artifacts/cleaning_artifacts.json")
print('  git commit -m "data: add cleaned processed splits + artifacts (DVC-tracked)"')
print("  git push")

Run in terminal:

  git add data/processed.dvc data/.gitignore artifacts/cleaning_artifacts.json
  git commit -m "data: add cleaned processed splits + artifacts (DVC-tracked)"
  git push


---
## Summary & Next Steps

| Done | Details |
|---|---|
| Nulls imputed | `total_bedrooms` global median (MCAR, p=0.3095) |
| `is_capped` flag | rows with target >= $500,001 |
| log1p applied | `total_rooms`, `total_bedrooms`, `population`, `households` |
| `median_income` | unchanged (no transform) |
| LOF flag | contamination=0.02, novelty=True |
| Files saved | `data/processed/` (DVC-tracked) |
| Artifacts saved | `artifacts/cleaning_artifacts.json` + `lof_model.pkl` |

